# Tests: función `preprocesamiento`

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "ftfy", "pandas", "numpy", "-q"])


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


0

In [2]:
import unittest
import pandas as pd
import numpy as np
import ftfy
import re

# Función bajo prueba
def preprocesamiento(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return ""
    texto = ftfy.fix_text(texto)
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'[^\w\s.,!?¡¿]', '', texto)
    texto = texto.replace('\n', ' ').replace('\r', '')
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip().lower()

## Suite 1 — Entradas vacías o inválidas

In [3]:
class TestEntradaInvalida(unittest.TestCase):

    def test_nan_retorna_cadena_vacia(self):
        self.assertEqual(preprocesamiento(float('nan')), "")

    def test_none_retorna_cadena_vacia(self):
        self.assertEqual(preprocesamiento(None), "")

    def test_entero_retorna_cadena_vacia(self):
        self.assertEqual(preprocesamiento(42), "")

    def test_lista_retorna_cadena_vacia(self):
        self.assertEqual(preprocesamiento(["hola"]), "")

    def test_cadena_vacia_retorna_cadena_vacia(self):
        self.assertEqual(preprocesamiento(""), "")


unittest.main(argv=[''], exit=False, verbosity=2)

test_cadena_vacia_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_cadena_vacia_retorna_cadena_vacia) ... ok
test_entero_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_entero_retorna_cadena_vacia) ... ok
test_lista_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_lista_retorna_cadena_vacia) ... ok
test_nan_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_nan_retorna_cadena_vacia) ... ok
test_none_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_none_retorna_cadena_vacia) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.002s

OK


## Suite 2 — Corrección de Mojibake (ftfy)

In [4]:
class TestMojibake(unittest.TestCase):

    def test_tilde_corrupta_se_corrige(self):
        # 'Ã¡' es la codificación mojibake de 'á'
        resultado = preprocesamiento("EstÃ¡ bien")
        self.assertIn("está bien", resultado)

    def test_enie_corrupta_se_corrige(self):
        # 'Ã\xb1' es mojibake de 'ñ'
        resultado = preprocesamiento("espaÃ\xb1ol")
        self.assertIn("español", resultado)

    def test_texto_limpio_no_se_altera(self):
        resultado = preprocesamiento("texto limpio")
        self.assertEqual(resultado, "texto limpio")


unittest.main(argv=[''], exit=False, verbosity=2)

test_cadena_vacia_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_cadena_vacia_retorna_cadena_vacia) ... ok
test_entero_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_entero_retorna_cadena_vacia) ... ok
test_lista_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_lista_retorna_cadena_vacia) ... ok
test_nan_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_nan_retorna_cadena_vacia) ... ok
test_none_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_none_retorna_cadena_vacia) ... ok
test_enie_corrupta_se_corrige (__main__.TestMojibake.test_enie_corrupta_se_corrige) ... ok
test_texto_limpio_no_se_altera (__main__.TestMojibake.test_texto_limpio_no_se_altera) ... ok
test_tilde_corrupta_se_corrige (__main__.TestMojibake.test_tilde_corrupta_se_corrige) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.003s

OK


## Suite 3 — Eliminación de URLs

In [5]:
class TestEliminacionURLs(unittest.TestCase):

    def test_url_http_eliminada(self):
        resultado = preprocesamiento("mira esto http://ejemplo.com increíble")
        self.assertNotIn("http", resultado)
        self.assertIn("increíble", resultado)

    def test_url_https_eliminada(self):
        resultado = preprocesamiento("visita https://ejemplo.com/articulo")
        self.assertNotIn("https", resultado)

    def test_url_www_eliminada(self):
        resultado = preprocesamiento("ver www.ejemplo.com para más info")
        self.assertNotIn("www", resultado)

    def test_texto_sin_url_no_cambia(self):
        resultado = preprocesamiento("hola mundo")
        self.assertEqual(resultado, "hola mundo")


unittest.main(argv=[''], exit=False, verbosity=2)

test_texto_sin_url_no_cambia (__main__.TestEliminacionURLs.test_texto_sin_url_no_cambia) ... ok
test_url_http_eliminada (__main__.TestEliminacionURLs.test_url_http_eliminada) ... ok
test_url_https_eliminada (__main__.TestEliminacionURLs.test_url_https_eliminada) ... ok
test_url_www_eliminada (__main__.TestEliminacionURLs.test_url_www_eliminada) ... ok
test_cadena_vacia_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_cadena_vacia_retorna_cadena_vacia) ... ok
test_entero_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_entero_retorna_cadena_vacia) ... ok
test_lista_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_lista_retorna_cadena_vacia) ... ok
test_nan_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_nan_retorna_cadena_vacia) ... ok
test_none_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_none_retorna_cadena_vacia) ... ok
test_enie_corrupta_se_corrige (__main__.TestMojibake.test_enie_corrupta_se_corrige) ... ok
test_texto_limpio_no_se_altera (__m

## Suite 4 — Eliminación de menciones (@usuario)

In [6]:
class TestEliminacionMenciones(unittest.TestCase):

    def test_mencion_simple_eliminada(self):
        resultado = preprocesamiento("hola @usuario como estás")
        self.assertNotIn("@usuario", resultado)
        self.assertIn("hola", resultado)

    def test_multiple_menciones_eliminadas(self):
        resultado = preprocesamiento("@user1 y @user2 se fueron")
        self.assertNotIn("@", resultado)
        self.assertIn("se fueron", resultado)

    def test_arroba_en_email_eliminada(self):
        # El regex elimina cualquier @palabra, incluyendo fragmentos de correos
        resultado = preprocesamiento("contacto en correo@dominio")
        self.assertNotIn("@dominio", resultado)


unittest.main(argv=[''], exit=False, verbosity=2)

test_arroba_en_email_eliminada (__main__.TestEliminacionMenciones.test_arroba_en_email_eliminada) ... ok
test_mencion_simple_eliminada (__main__.TestEliminacionMenciones.test_mencion_simple_eliminada) ... ok
test_multiple_menciones_eliminadas (__main__.TestEliminacionMenciones.test_multiple_menciones_eliminadas) ... ok
test_texto_sin_url_no_cambia (__main__.TestEliminacionURLs.test_texto_sin_url_no_cambia) ... ok
test_url_http_eliminada (__main__.TestEliminacionURLs.test_url_http_eliminada) ... ok
test_url_https_eliminada (__main__.TestEliminacionURLs.test_url_https_eliminada) ... ok
test_url_www_eliminada (__main__.TestEliminacionURLs.test_url_www_eliminada) ... ok
test_cadena_vacia_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_cadena_vacia_retorna_cadena_vacia) ... ok
test_entero_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_entero_retorna_cadena_vacia) ... ok
test_lista_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_lista_retorna_cadena_vacia) ... ok
tes

## Suite 5 — Preservación de puntuación clínica relevante

In [7]:
class TestPuntuacion(unittest.TestCase):

    def test_punto_se_preserva(self):
        resultado = preprocesamiento("No como. Nunca.")
        self.assertIn(".", resultado)

    def test_exclamacion_se_preserva(self):
        resultado = preprocesamiento("¡Qué hambre!")
        self.assertIn("!", resultado)

    def test_interrogacion_se_preserva(self):
        resultado = preprocesamiento("¿Por qué como tanto?")
        self.assertIn("?", resultado)
        self.assertIn("¿", resultado)

    def test_coma_se_preserva(self):
        resultado = preprocesamiento("hambre, sed, cansancio")
        self.assertIn(",", resultado)

    def test_emojis_eliminados(self):
        resultado = preprocesamiento("me siento mal 💔")
        self.assertNotIn("💔", resultado)

    def test_hashtag_elimina_simbolo_preserva_texto(self):
        # El # no está en el conjunto permitido, pero las letras sí
        resultado = preprocesamiento("#HastaLosHuesos es una etiqueta")
        self.assertNotIn("#", resultado)
        self.assertIn("hastaloshuesos", resultado)


unittest.main(argv=[''], exit=False, verbosity=2)

test_arroba_en_email_eliminada (__main__.TestEliminacionMenciones.test_arroba_en_email_eliminada) ... ok
test_mencion_simple_eliminada (__main__.TestEliminacionMenciones.test_mencion_simple_eliminada) ... ok
test_multiple_menciones_eliminadas (__main__.TestEliminacionMenciones.test_multiple_menciones_eliminadas) ... ok
test_texto_sin_url_no_cambia (__main__.TestEliminacionURLs.test_texto_sin_url_no_cambia) ... ok
test_url_http_eliminada (__main__.TestEliminacionURLs.test_url_http_eliminada) ... ok
test_url_https_eliminada (__main__.TestEliminacionURLs.test_url_https_eliminada) ... ok
test_url_www_eliminada (__main__.TestEliminacionURLs.test_url_www_eliminada) ... ok
test_cadena_vacia_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_cadena_vacia_retorna_cadena_vacia) ... ok
test_entero_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_entero_retorna_cadena_vacia) ... ok
test_lista_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_lista_retorna_cadena_vacia) ... ok
tes

## Suite 6 — Normalización de espacios y saltos de línea

In [8]:
class TestEspacios(unittest.TestCase):

    def test_salto_de_linea_reemplazado_por_espacio(self):
        resultado = preprocesamiento("hola\nmundo")
        self.assertNotIn("\n", resultado)
        self.assertIn("hola mundo", resultado)

    def test_retorno_de_carro_eliminado(self):
        resultado = preprocesamiento("hola\rmundo")
        self.assertNotIn("\r", resultado)

    def test_espacios_multiples_colapsados(self):
        resultado = preprocesamiento("hola    mundo")
        self.assertEqual(resultado, "hola mundo")

    def test_espacios_al_inicio_y_fin_eliminados(self):
        resultado = preprocesamiento("  hola mundo  ")
        self.assertEqual(resultado, "hola mundo")


unittest.main(argv=[''], exit=False, verbosity=2)

test_arroba_en_email_eliminada (__main__.TestEliminacionMenciones.test_arroba_en_email_eliminada) ... ok
test_mencion_simple_eliminada (__main__.TestEliminacionMenciones.test_mencion_simple_eliminada) ... ok
test_multiple_menciones_eliminadas (__main__.TestEliminacionMenciones.test_multiple_menciones_eliminadas) ... ok
test_texto_sin_url_no_cambia (__main__.TestEliminacionURLs.test_texto_sin_url_no_cambia) ... ok
test_url_http_eliminada (__main__.TestEliminacionURLs.test_url_http_eliminada) ... ok
test_url_https_eliminada (__main__.TestEliminacionURLs.test_url_https_eliminada) ... ok
test_url_www_eliminada (__main__.TestEliminacionURLs.test_url_www_eliminada) ... ok
test_cadena_vacia_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_cadena_vacia_retorna_cadena_vacia) ... ok
test_entero_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_entero_retorna_cadena_vacia) ... ok
test_lista_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_lista_retorna_cadena_vacia) ... ok
tes

## Suite 7 — Conversión a minúsculas

In [9]:
class TestMinusculas(unittest.TestCase):

    def test_mayusculas_convertidas(self):
        resultado = preprocesamiento("ANOREXIA")
        self.assertEqual(resultado, "anorexia")

    def test_mixto_convertido(self):
        resultado = preprocesamiento("No Como Nada")
        self.assertEqual(resultado, "no como nada")

    def test_ya_minusculas_no_cambia(self):
        resultado = preprocesamiento("todo en minúsculas")
        self.assertEqual(resultado, "todo en minúsculas")


unittest.main(argv=[''], exit=False, verbosity=2)

test_arroba_en_email_eliminada (__main__.TestEliminacionMenciones.test_arroba_en_email_eliminada) ... ok
test_mencion_simple_eliminada (__main__.TestEliminacionMenciones.test_mencion_simple_eliminada) ... ok
test_multiple_menciones_eliminadas (__main__.TestEliminacionMenciones.test_multiple_menciones_eliminadas) ... ok
test_texto_sin_url_no_cambia (__main__.TestEliminacionURLs.test_texto_sin_url_no_cambia) ... ok
test_url_http_eliminada (__main__.TestEliminacionURLs.test_url_http_eliminada) ... ok
test_url_https_eliminada (__main__.TestEliminacionURLs.test_url_https_eliminada) ... ok
test_url_www_eliminada (__main__.TestEliminacionURLs.test_url_www_eliminada) ... ok
test_cadena_vacia_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_cadena_vacia_retorna_cadena_vacia) ... ok
test_entero_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_entero_retorna_cadena_vacia) ... ok
test_lista_retorna_cadena_vacia (__main__.TestEntradaInvalida.test_lista_retorna_cadena_vacia) ... ok
tes

## Suite 8 — Casos reales del dataset (tweets con Mojibake)

In [10]:
class TestCasosRealesDataset(unittest.TestCase):

    def test_tweet_mojibake_azucar(self):
        # Ejemplo real del dataset: 'azÃºcar' → 'azúcar'
        resultado = preprocesamiento("Cheesecake saludable sin azÃºcar y sin lactosa")
        self.assertIn("azúcar", resultado)
        self.assertNotIn("azÃºcar", resultado)

    def test_tweet_mojibake_frio(self):
        # 'frÃ­o' → 'frío'
        resultado = preprocesamiento("no sentÃ­a mi cuerpo tan frÃ­o")
        self.assertIn("frío", resultado)

    def test_tweet_con_hashtag_y_emoji(self):
        resultado = preprocesamiento("ser como ellas ♡♡\n  #HastaLosHuesos")
        self.assertNotIn("♡", resultado)
        self.assertNotIn("#", resultado)
        self.assertNotIn("\n", resultado)
        self.assertIn("hastaloshuesos", resultado)


unittest.main(argv=[''], exit=False, verbosity=2)

test_tweet_con_hashtag_y_emoji (__main__.TestCasosRealesDataset.test_tweet_con_hashtag_y_emoji) ... ok
test_tweet_mojibake_azucar (__main__.TestCasosRealesDataset.test_tweet_mojibake_azucar) ... ok
test_tweet_mojibake_frio (__main__.TestCasosRealesDataset.test_tweet_mojibake_frio) ... ok
test_arroba_en_email_eliminada (__main__.TestEliminacionMenciones.test_arroba_en_email_eliminada) ... ok
test_mencion_simple_eliminada (__main__.TestEliminacionMenciones.test_mencion_simple_eliminada) ... ok
test_multiple_menciones_eliminadas (__main__.TestEliminacionMenciones.test_multiple_menciones_eliminadas) ... ok
test_texto_sin_url_no_cambia (__main__.TestEliminacionURLs.test_texto_sin_url_no_cambia) ... ok
test_url_http_eliminada (__main__.TestEliminacionURLs.test_url_http_eliminada) ... ok
test_url_https_eliminada (__main__.TestEliminacionURLs.test_url_https_eliminada) ... ok
test_url_www_eliminada (__main__.TestEliminacionURLs.test_url_www_eliminada) ... ok
test_cadena_vacia_retorna_cadena_vac